In [4]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import openeo

In [5]:
connection = openeo.connect("https://openeo.dataspace.copernicus.eu")

In [6]:
print(connection.list_collection_ids())

['SENTINEL1_GRD', 'COPERNICUS_30', 'SENTINEL3_OLCI_L1B', 'SENTINEL3_SLSTR', 'SENTINEL_5P_L2', 'LANDSAT_BIMONTHLY_MOSAIC', 'SENTINEL3_SYN_L2_SYN', 'SENTINEL3_SLSTR_L2_LST', 'SENTINEL5P_L2_CO', 'SENTINEL5P_L2_NO2', 'SENTINEL5P_L2_CH4', 'COPERNICUS_VEGETATION_PHENOLOGY_PRODUCTIVITY_10M_SEASON1', 'COPERNICUS_VEGETATION_PHENOLOGY_PRODUCTIVITY_10M_SEASON2', 'COPERNICUS_PLANT_PHENOLOGY_INDEX', 'ESA_WORLDCOVER_10M_2020_V1', 'ESA_WORLDCOVER_10M_2021_V2', 'SENTINEL1_GLOBAL_MOSAICS', 'SENTINEL2_L2A', 'SENTINEL2_L1C', 'SENTINEL3_OLCI_L2_LAND', 'SENTINEL3_OLCI_L2_WATER', 'SENTINEL3_SYN_L2_AOD', 'CLMS_LCM_GLOBAL_10M_YEARLY_V1', 'CLMS_TCD_PANTROPICAL_10M_YEARLY_V1', 'CLMS_BA_GLOBAL_300M_DAILY_V4', 'CLMS_NDVI_GLOBAL_300M_10DAILY_V3', 'CLMS_ETA_GLOBAL_300M_10DAILY_V1', 'CLMS_HF_GLOBAL_300M_DAILY_V1', 'CLMS_LSP_GLOBAL_300M_YEARLY_V2', 'CLMS_SSM_EUROPE_1KM_DAILY_V1', 'CLMS_NPP_GLOBAL_300M_10DAILY_V2', 'CLMS_LST_TCI_GLOBAL_3KM_10DAILY_V3', 'CLMS_FAPAR_GLOBAL_1KM_10DAILY_V2', 'CLMS_FCOVER_GLOBAL_1KM_10DAIL

In [7]:
print(connection.describe_collection("SENTINEL2_L2A"))

{'bands': [{'name': 'B01'}, {'name': 'B02'}, {'name': 'B03'}, {'name': 'B04'}, {'name': 'B05'}, {'name': 'B06'}, {'name': 'B07'}, {'name': 'B08'}, {'name': 'B8A'}, {'name': 'B09'}, {'name': 'B11'}, {'name': 'B12'}, {'name': 'WVP'}, {'name': 'AOT'}, {'name': 'SCL'}, {'name': 'sunAzimuthAngles'}, {'name': 'sunZenithAngles'}, {'name': 'viewAzimuthMean'}, {'name': 'viewZenithMean'}, {'name': 'CLD'}, {'name': 'SNW'}], 'ceosard:specification': 'SR', 'ceosard:specification_version': '5.0.1', 'ceosard:type': 'optical', 'cube:dimensions': {'bands': {'type': 'bands', 'values': ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B11', 'B12', 'WVP', 'AOT', 'SCL', 'sunAzimuthAngles', 'sunZenithAngles', 'viewAzimuthMean', 'viewZenithMean', 'CLD', 'SNW']}, 't': {'extent': ['2015-06-27T10:25:31Z', None], 'type': 'temporal'}, 'x': {'axis': 'x', 'extent': [-180, 180], 'reference_system': {'$schema': 'https://proj.org/schemas/v0.2/projjson.schema.json', 'area': 'World', 'bbox': {'east

In [2]:
raster_path = "Sentinel2/Sentinel2_JawaTimur.tif"

In [9]:
sample_paths = {
    "Sawah": "materi/Sampel Sawah/input.shp",
    "Bangunan": "materi/Sampel Non-sawah/input.shp",
    "Danau": "materi/Danau/input.shp",
    "Laut": "materi/Laut/input.shp",
    "Hutan": "materi/lahan hijau/input.shp",
    "Mangrove": "materi/Mangrov/input.shp"
}

In [10]:
samples = []

for label, path in sample_paths.items():
    gdf = gpd.read_file(path)
    gdf["label"] = label
    samples.append(gdf)

samples_gdf = gpd.GeoDataFrame(
    pd.concat(samples, ignore_index=True),
    crs=samples[0].crs
)

print("Total sampel:", len(samples_gdf))
print("\nJumlah setiap kelas:")
print(samples_gdf["label"].value_counts())

Total sampel: 256

Jumlah setiap kelas:
label
Sawah       50
Bangunan    50
Laut        50
Hutan       50
Danau       31
Mangrove    25
Name: count, dtype: int64


In [11]:
print("CRS sampel:")
print(samples_gdf.crs)

CRS sampel:
EPSG:4326


In [12]:
minx, miny, maxx, maxy = samples_gdf.total_bounds

print("Bounds:")
print("Min X:", minx)
print("Min Y:", miny)
print("Max X:", maxx)
print("Max Y:", maxy)

Bounds:
Min X: 112.0487761
Min Y: -7.5209009
Max X: 113.0503799
Max Y: -7.0437618


In [13]:
connection = openeo.connect(
    "https://openeo.dataspace.copernicus.eu"
)

connection.authenticate_oidc()

Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

In [36]:
bands = [
    "B02",
    "B03",
    "B04",
    "B05",
    "B06",
    "B07",
    "B08",
    "B8A",
    "B11",
    "B12"
]

datacube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent={
        "west": minx,
        "south": miny,
        "east": maxx,
        "north": maxy
    },
    temporal_extent=[
        "2025-01-01",
        "2025-01-31"
    ],
    bands=bands
)

In [37]:
features_median = features_10m.reduce_dimension(
    dimension="t",
    reducer="median"
)

In [38]:
# NDVI
ndvi = (
    (datacube.band("B08") - datacube.band("B04")) /
    (datacube.band("B08") + datacube.band("B04"))
)

# NDWI
ndwi = (
    (datacube.band("B03") - datacube.band("B08")) /
    (datacube.band("B03") + datacube.band("B08"))
)

# NDBI
ndbi = (
    (datacube.band("B11") - datacube.band("B08")) /
    (datacube.band("B11") + datacube.band("B08"))
)

# NDMI
ndmi = (
    (datacube.band("B08") - datacube.band("B11")) /
    (datacube.band("B08") + datacube.band("B11"))
)

# MNDWI
mndwi = (
    (datacube.band("B03") - datacube.band("B11")) /
    (datacube.band("B03") + datacube.band("B11"))
)

In [39]:
ndvi = ndvi.add_dimension(
    name="bands",
    label="NDVI",
    type="bands"
)

ndwi = ndwi.add_dimension(
    name="bands",
    label="NDWI",
    type="bands"
)

ndbi = ndbi.add_dimension(
    name="bands",
    label="NDBI",
    type="bands"
)

ndmi = ndmi.add_dimension(
    name="bands",
    label="NDMI",
    type="bands"
)

mndwi = mndwi.add_dimension(
    name="bands",
    label="MNDWI",
    type="bands"
)

In [40]:
features = datacube.merge_cubes(ndvi)
features = features.merge_cubes(ndwi)
features = features.merge_cubes(ndbi)
features = features.merge_cubes(ndmi)
features = features.merge_cubes(mndwi)

In [42]:
features_10m = features.resample_spatial(
    resolution=10,
    projection="EPSG:32749"
)

In [43]:
import os

os.makedirs("hasil", exist_ok=True)

job = features_median.execute_batch(
    outputfile="hasil/Sentinel2_JawaTimur.tif",
    title="Sentinel-2 Jawa Timur Land Classification",
    out_format="GTiff"
)

0:00:00 Job 'j-2610081812234b83a1446b24a0ffb888': send 'start'
0:00:03 Job 'j-2610081812234b83a1446b24a0ffb888': queued (progress 0%)
0:00:08 Job 'j-2610081812234b83a1446b24a0ffb888': queued (progress 0%)
0:00:15 Job 'j-2610081812234b83a1446b24a0ffb888': queued (progress 0%)
0:00:23 Job 'j-2610081812234b83a1446b24a0ffb888': queued (progress 0%)
0:00:33 Job 'j-2610081812234b83a1446b24a0ffb888': queued (progress 0%)
0:00:46 Job 'j-2610081812234b83a1446b24a0ffb888': queued (progress 0%)
0:01:02 Job 'j-2610081812234b83a1446b24a0ffb888': running (progress N/A)
0:01:21 Job 'j-2610081812234b83a1446b24a0ffb888': running (progress N/A)
0:01:46 Job 'j-2610081812234b83a1446b24a0ffb888': running (progress N/A)
0:02:16 Job 'j-2610081812234b83a1446b24a0ffb888': running (progress N/A)
0:02:54 Job 'j-2610081812234b83a1446b24a0ffb888': running (progress N/A)
0:03:41 Job 'j-2610081812234b83a1446b24a0ffb888': running (progress N/A)
0:04:39 Job 'j-2610081812234b83a1446b24a0ffb888': running (progress N/A)


JobFailedException: Batch job 'j-2610081812234b83a1446b24a0ffb888' didn't finish successfully. Status: error (after 0:53:03).

In [44]:
for label in samples_gdf["label"].unique():
    gdf = samples_gdf[samples_gdf["label"] == label]

    minx, miny, maxx, maxy = gdf.total_bounds

    print("\n", label)
    print("Bounds :", minx, miny, maxx, maxy)
    print("Lebar  :", maxx - minx, "meter")
    print("Tinggi :", maxy - miny, "meter")
    print(
        "Luas bbox :",
        round((maxx - minx) * (maxy - miny) / 1_000_000, 2),
        "km²"
    )


 Sawah
Bounds : 112.1912248 -7.2066008 112.1955727 -7.2025308
Lebar  : 0.004347899999999072 meter
Tinggi : 0.004070000000000462 meter
Luas bbox : 0.0 km²

 Bangunan
Bounds : 112.1881514 -7.2227551 112.1909746 -7.2204264
Lebar  : 0.0028232000000087965 meter
Tinggi : 0.0023286999999996283 meter
Luas bbox : 0.0 km²

 Danau
Bounds : 112.203671 -7.3839902 112.7417692 -7.0437618
Lebar  : 0.5380981999999932 meter
Tinggi : 0.3402284 meter
Luas bbox : 0.0 km²

 Laut
Bounds : 112.8511344 -7.5209009 113.0503799 -7.4398392
Lebar  : 0.1992454999999893 meter
Tinggi : 0.08106170000000024 meter
Luas bbox : 0.0 km²

 Hutan
Bounds : 112.0487761 -7.3822521 112.0608883 -7.3759691
Lebar  : 0.012112200000004236 meter
Tinggi : 0.0062829999999998165 meter
Luas bbox : 0.0 km²

 Mangrove
Bounds : 112.7125455 -7.2857798 112.846131 -7.1974972
Lebar  : 0.13358549999999525 meter
Tinggi : 0.08828260000000032 meter
Luas bbox : 0.0 km²


In [45]:
samples_utm = samples_gdf.to_crs("EPSG:32749")

print(samples_utm.crs)

EPSG:32749


In [46]:
minx, miny, maxx, maxy = samples_utm.total_bounds

print("MIN X :", minx)
print("MIN Y :", miny)
print("MAX X :", maxx)
print("MAX Y :", maxy)

print("\nLebar :", maxx - minx, "meter")
print("Tinggi:", maxy - miny, "meter")

print(
    "\nLuas bounding box:",
    round((maxx - minx) * (maxy - miny) / 1_000_000, 2),
    "km²"
)

MIN X : 615748.6140634046
MIN Y : 9168147.46607959
MAX X : 726291.5521893258
MAX Y : 9221055.37320133

Lebar : 110542.93812592118 meter
Tinggi: 52907.90712173842 meter

Luas bounding box: 5848.6 km²


In [47]:
for label in samples_utm["label"].unique():
    gdf = samples_utm[samples_utm["label"] == label]

    minx, miny, maxx, maxy = gdf.total_bounds

    print("\n", label)
    print("Lebar :", round(maxx - minx, 2), "meter")
    print("Tinggi:", round(maxy - miny, 2), "meter")
    print(
        "Luas bbox:",
        round((maxx - minx) * (maxy - miny) / 1_000_000, 2),
        "km²"
    )


 Sawah
Lebar : 480.55 meter
Tinggi: 449.28 meter
Luas bbox: 0.22 km²

 Bangunan
Lebar : 311.57 meter
Tinggi: 257.89 meter
Luas bbox: 0.08 km²

 Danau
Lebar : 59498.65 meter
Tinggi: 37495.93 meter
Luas bbox: 2230.96 km²

 Laut
Lebar : 22000.74 meter
Tinggi: 9002.42 meter
Luas bbox: 198.06 km²

 Hutan
Lebar : 1336.55 meter
Tinggi: 695.45 meter
Luas bbox: 0.93 km²

 Mangrove
Lebar : 14720.07 meter
Tinggi: 9807.6 meter
Luas bbox: 144.37 km²


In [48]:
samples_utm = samples_gdf.to_crs("EPSG:32749")

for label in samples_utm["label"].unique():
    gdf = samples_utm[samples_utm["label"] == label]
    minx, miny, maxx, maxy = gdf.total_bounds

    print(
        label,
        "|",
        round(maxx - minx, 2), "x",
        round(maxy - miny, 2),
        "meter"
    )

Sawah | 480.55 x 449.28 meter
Bangunan | 311.57 x 257.89 meter
Danau | 59498.65 x 37495.93 meter
Laut | 22000.74 x 9002.42 meter
Hutan | 1336.55 x 695.45 meter
Mangrove | 14720.07 x 9807.6 meter


In [50]:
samples_wgs84 = samples_gdf.to_crs("EPSG:4326")

print(samples_wgs84.crs)
print(len(samples_wgs84))

EPSG:4326
256


In [51]:
import json

sample_geojson = json.loads(
    samples_wgs84.to_json()
)

print(sample_geojson["type"])
print(len(sample_geojson["features"]))

FeatureCollection
256


In [52]:
minx, miny, maxx, maxy = samples_wgs84.total_bounds

print("West :", minx)
print("South:", miny)
print("East :", maxx)
print("North:", maxy)

West : 112.0487761
South: -7.5209009
East : 113.0503799
North: -7.0437618


In [53]:
bands = [
    "B02",
    "B03",
    "B04",
    "B05",
    "B06",
    "B07",
    "B08",
    "B8A",
    "B11",
    "B12"
]

datacube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent={
        "west": minx,
        "south": miny,
        "east": maxx,
        "north": maxy
    },
    temporal_extent=[
        "2025-08-01",
        "2025-09-01"
    ],
    bands=bands,
    max_cloud_cover=20
)

In [54]:
datacube_median = datacube.reduce_dimension(
    dimension="t",
    reducer="median"
)

In [55]:
ndvi = (
    (datacube_median.band("B08") - datacube_median.band("B04")) /
    (datacube_median.band("B08") + datacube_median.band("B04"))
)

ndwi = (
    (datacube_median.band("B03") - datacube_median.band("B08")) /
    (datacube_median.band("B03") + datacube_median.band("B08"))
)

ndbi = (
    (datacube_median.band("B11") - datacube_median.band("B08")) /
    (datacube_median.band("B11") + datacube_median.band("B08"))
)

ndmi = (
    (datacube_median.band("B08") - datacube_median.band("B11")) /
    (datacube_median.band("B08") + datacube_median.band("B11"))
)

mndwi = (
    (datacube_median.band("B03") - datacube_median.band("B11")) /
    (datacube_median.band("B03") + datacube_median.band("B11"))
)

In [56]:
ndvi = ndvi.add_dimension(
    name="bands",
    label="NDVI",
    type="bands"
)

ndwi = ndwi.add_dimension(
    name="bands",
    label="NDWI",
    type="bands"
)

ndbi = ndbi.add_dimension(
    name="bands",
    label="NDBI",
    type="bands"
)

ndmi = ndmi.add_dimension(
    name="bands",
    label="NDMI",
    type="bands"
)

mndwi = mndwi.add_dimension(
    name="bands",
    label="MNDWI",
    type="bands"
)

In [57]:
features = datacube_median.merge_cubes(ndvi)
features = features.merge_cubes(ndwi)
features = features.merge_cubes(ndbi)
features = features.merge_cubes(ndmi)
features = features.merge_cubes(mndwi)

In [58]:
print(features.metadata.band_names)

['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12', 'NDVI', 'NDWI', 'NDBI', 'NDMI', 'MNDWI']


In [59]:
sample_features = features.aggregate_spatial(
    geometries=sample_geojson,
    reducer="mean"
)

In [60]:
os.makedirs("hasil", exist_ok=True)

job = sample_features.execute_batch(
    outputfile="hasil/fitur_sampel_sentinel.geojson",
    title="Ekstraksi Fitur Sentinel-2 Sampel"
)

Preflight process graph validation raised: [Internal] Unexpected error during 'aggregate_spatial' (node id 'aggregatespatial1'): GEOSException('TopologyException: side location conflict at 112.19475456923077 -7.2045458453109035. This can occur if the input geometry is invalid.'). The process had these arguments: {'data': DryRunDataCube(metadata=CollectionMetadata(dimension_names=['x', 'y', 'bands'], band_names=['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B11', 'B12', 'WVP', 'AOT', 'SCL', 'sunAzimuthAngles', 'sunZenithAngles', 'viewAzimuthMean', 'viewZenithMean', 'CLD', 'SNW'])), 'geometries': {'type': 'FeatureCollection', 'features': [{'id': '0', 'type': 'Feature', 'properties': {'FID': 0, 'label': 'Sawah'}, 'geometry': {'type': 'Polygon', 'coordinates': [[[112.1947603, -7.2044328], [112.194753, -7.2045768], [112.1947555, -7.2045792], [112.1947482, -7.2043176], [112.1950167, -7.2043656], [112.1952948, -7.2044616], [112.1952996, -7.2045768], [112.1947579, -7.2

0:00:00 Job 'j-26100819303648a68d62294d85aed7db': send 'start'


OpenEoApiError: [500] Internal: Unexpected error during 'aggregate_spatial' (node id 'aggregatespatial1'): GEOSException('TopologyException: side location conflict at 112.19475456923077 -7.2045458453109035. This can occur if the input geometry is invalid.'). The process had these arguments: {'data': DryRunDataCube(metadata=CollectionMetadata(dimension_names=['x', 'y', 'bands'], band_names=['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B11', 'B12', 'WVP', 'AOT', 'SCL', 'sunAzimuthAngles', 'sunZenithAngles', 'viewAzimuthMean', 'viewZenithMean', 'CLD', 'SNW'])), 'geometries': {'type': 'FeatureCollection', 'features': [{'id': '0', 'type': 'Feature', 'properties': {'FID': 0, 'label': 'Sawah'}, 'geometry': {'type': 'Polygon', 'coordinates': [[[112.1947603, -7.2044328], [112.194753, -7.2045768], [112.1947555, -7.2045792], [112.1947482, -7.2043176], [112.1950167, -7.2043656], [112.1952948, -7.2044616], [112.1952996, -7.2045768], [112.1947579, -7.2045792], [112.1947603,

In [61]:
import geopandas as gpd
import json
import os

# Salin data sampel
samples_wgs84 = samples_gdf.to_crs("EPSG:4326").copy()

print("Jumlah awal:", len(samples_wgs84))
print("CRS:", samples_wgs84.crs)

# Cek geometry invalid
print("Geometry invalid:", (~samples_wgs84.geometry.is_valid).sum())
print("Geometry kosong:", samples_wgs84.geometry.is_empty.sum())


Jumlah awal: 256
CRS: EPSG:4326
Geometry invalid: 1
Geometry kosong: 0


In [62]:
from shapely.validation import make_valid

# Perbaiki geometry yang invalid
samples_wgs84["geometry"] = samples_wgs84.geometry.apply(
    lambda geom: make_valid(geom) if geom is not None else geom
)

# Kalau masih ada geometry invalid, gunakan buffer(0)
samples_wgs84["geometry"] = samples_wgs84.geometry.apply(
    lambda geom: geom.buffer(0) if geom is not None and not geom.is_valid else geom
)

# Hapus geometry kosong/null
samples_wgs84 = samples_wgs84[
    samples_wgs84.geometry.notna() &
    ~samples_wgs84.geometry.is_empty
].copy()

# Cek kembali
print("Jumlah setelah perbaikan:", len(samples_wgs84))
print("Geometry invalid:", (~samples_wgs84.geometry.is_valid).sum())
print("Geometry kosong:", samples_wgs84.geometry.is_empty.sum())

Jumlah setelah perbaikan: 256
Geometry invalid: 0
Geometry kosong: 0


In [63]:
samples_wgs84 = samples_wgs84.reset_index(drop=True)

samples_wgs84["sample_id"] = range(len(samples_wgs84))

print(samples_wgs84[["sample_id", "label"]].head())
print(samples_wgs84["label"].value_counts())

   sample_id  label
0          0  Sawah
1          1  Sawah
2          2  Sawah
3          3  Sawah
4          4  Sawah
label
Sawah       50
Bangunan    50
Laut        50
Hutan       50
Danau       31
Mangrove    25
Name: count, dtype: int64


In [66]:
sample_features = features.aggregate_spatial(
    geometries=sample_geojson,
    reducer="mean"
)

print("aggregate_spatial berhasil dibuat.")

aggregate_spatial berhasil dibuat.


In [67]:
os.makedirs("hasil", exist_ok=True)

job = sample_features.execute_batch(
    outputfile="hasil/fitur_sampel_sentinel.geojson",
    title="Ekstraksi Fitur Sentinel-2 Sampel"
)

0:00:00 Job 'j-26100819334742fabe22565710a37ed8': send 'start'
0:00:05 Job 'j-26100819334742fabe22565710a37ed8': queued (progress 0%)
0:00:12 Job 'j-26100819334742fabe22565710a37ed8': queued (progress 0%)
0:00:20 Job 'j-26100819334742fabe22565710a37ed8': queued (progress 0%)
0:00:29 Job 'j-26100819334742fabe22565710a37ed8': running (progress N/A)
0:00:42 Job 'j-26100819334742fabe22565710a37ed8': running (progress N/A)
0:00:57 Job 'j-26100819334742fabe22565710a37ed8': running (progress N/A)
0:01:14 Job 'j-26100819334742fabe22565710a37ed8': running (progress N/A)
0:01:35 Job 'j-26100819334742fabe22565710a37ed8': running (progress N/A)
0:02:01 Job 'j-26100819334742fabe22565710a37ed8': running (progress N/A)
0:02:32 Job 'j-26100819334742fabe22565710a37ed8': running (progress N/A)
0:03:10 Job 'j-26100819334742fabe22565710a37ed8': running (progress N/A)
0:03:58 Job 'j-26100819334742fabe22565710a37ed8': error (progress N/A)
Your batch job 'j-26100819334742fabe22565710a37ed8' failed. Error log

JobFailedException: Batch job 'j-26100819334742fabe22565710a37ed8' didn't finish successfully. Status: error (after 0:03:59).

In [68]:

import geopandas as gpd
import pandas as pd
import numpy as np

from sklearn.cluster import DBSCAN
from shapely.validation import make_valid
from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import unary_union

# 1. Samakan CRS
samples = sampel.to_crs("EPSG:4326").copy()

# 2. Perbaiki geometri
def clean_geometry(geom):
    if geom is None or geom.is_empty:
        return None

    geom = make_valid(geom)

    if geom.geom_type == "GeometryCollection":
        polygons = [
            g for g in geom.geoms
            if g.geom_type in ["Polygon", "MultiPolygon"]
        ]

        if not polygons:
            return None

        geom = unary_union(polygons)

    if geom.geom_type not in ["Polygon", "MultiPolygon"]:
        return None

    if not geom.is_valid:
        geom = geom.buffer(0)

    if geom.is_empty or not geom.is_valid:
        return None

    return geom

samples["geometry"] = samples.geometry.apply(clean_geometry)
samples = samples.dropna(subset=["geometry"]).copy()
samples = samples[~samples.geometry.is_empty].copy()
samples = samples.reset_index(drop=True)

# 3. Buat ID sampel
samples["sample_id"] = range(len(samples))

# 4. Hitung centroid dalam satuan meter
samples_utm = samples.to_crs("EPSG:32749")
centroids = samples_utm.geometry.centroid

coords = np.column_stack([centroids.x, centroids.y])

# 5. Kelompokkan lokasi dengan jarak awal 20 km
model_cluster = DBSCAN(eps=20000, min_samples=1)
samples["cluster"] = model_cluster.fit_predict(coords) + 1

# 6. Periksa hasil
print("Jumlah sampel valid:", len(samples))
print("Jumlah kelompok:", samples["cluster"].nunique())

print("\nJumlah sampel per kelas:")
print(samples["label"].value_counts())

print("\nJumlah sampel per kelompok:")
print(samples.groupby("cluster").size())

print("\nDistribusi kelas per kelompok:")
print(pd.crosstab(samples["cluster"], samples["label"]))


NameError: name 'sampel' is not defined

In [70]:

import geopandas as gpd
import pandas as pd

# Baca keenam shapefile
sawah = gpd.read_file("materi/Sampel Sawah/input.shp")
bangunan = gpd.read_file("materi/Sampel Non-sawah/input.shp")
danau = gpd.read_file("materi/danau/input.shp")
laut = gpd.read_file("materi/laut_lepas/input.shp")
hutan = gpd.read_file("materi/lahan hijau/input.shp")
mangrove = gpd.read_file("materi/mangrove/input.shp")

# Berikan label kelas
sawah["label"] = "sawah"
bangunan["label"] = "bangunan"
danau["label"] = "danau"
laut["label"] = "laut"
hutan["label"] = "hutan"
mangrove["label"] = "mangrove"

# Samakan CRS seluruh data
crs_target = sawah.crs

bangunan = bangunan.to_crs(crs_target)
danau = danau.to_crs(crs_target)
laut = laut.to_crs(crs_target)
hutan = hutan.to_crs(crs_target)
mangrove = mangrove.to_crs(crs_target)

# Gabungkan seluruh sampel
sampel = gpd.GeoDataFrame(
    pd.concat(
        [sawah, bangunan, danau, laut, hutan, mangrove],
        ignore_index=True
    ),
    geometry="geometry",
    crs=crs_target
)

print("Total sampel:", len(sampel))
print("\nJumlah sampel per kelas:")
print(sampel["label"].value_counts())


Total sampel: 291

Jumlah sampel per kelas:
label
laut        60
sawah       50
bangunan    50
hutan       50
mangrove    50
danau       31
Name: count, dtype: int64


In [73]:

import numpy as np
from sklearn.cluster import DBSCAN

# Pastikan data sampel menggunakan CRS metrik
samples = samples.to_crs("EPSG:4326").copy()
samples_utm = samples.to_crs("EPSG:32749")

# Hitung titik tengah setiap polygon dalam satuan meter
centroids = samples_utm.geometry.centroid

coords = np.column_stack([
    centroids.x,
    centroids.y
])

# Kelompokkan sampel berdasarkan jarak 20 km
dbscan = DBSCAN(
    eps=20000,
    min_samples=1
)

samples["cluster"] = dbscan.fit_predict(coords) + 1

# Tampilkan hasil
print("Total sampel:", len(samples))
print("Jumlah kelompok:", samples["cluster"].nunique())

print("\nJumlah sampel per kelompok:")
print(samples.groupby("cluster").size())

print("\nDistribusi kelas per kelompok:")
print(pd.crosstab(samples["cluster"], samples["label"]))


Total sampel: 291
Jumlah kelompok: 3

Jumlah sampel per kelompok:
cluster
1    235
2     50
3      6
dtype: int64

Distribusi kelas per kelompok:
label    bangunan  danau  hutan  laut  mangrove  sawah
cluster                                               
1              50     31      0    60        44     50
2               0      0     50     0         0      0
3               0      0      0     0         6      0


In [74]:
samples = sampel.to_crs("EPSG:4326").copy()

In [76]:

print("Jumlah sampel:", len(samples))

print("\nJumlah geometri duplikat:")
print(samples.geometry.duplicated().sum())

print("\nJumlah geometri per kelas:")
print(samples.groupby("label").size())

print("\nJumlah ID sampel duplikat:")
if "sample_id" in samples.columns:
    print(samples["sample_id"].duplicated().sum())
else:
    print("Kolom sample_id belum tersedia.")


Jumlah sampel: 291

Jumlah geometri duplikat:
0

Jumlah geometri per kelas:
label
bangunan    50
danau       31
hutan       50
laut        60
mangrove    50
sawah       50
dtype: int64

Jumlah ID sampel duplikat:
Kolom sample_id belum tersedia.


In [77]:

# Buat ID unik untuk setiap sampel
samples = samples.reset_index(drop=True).copy()
samples["sample_id"] = range(len(samples))

# Periksa jumlah fitur setiap shapefile sumber
print("Jumlah fitur pada shapefile sumber:")

for nama, data in [
    ("Sawah", sawah),
    ("Bangunan", bangunan),
    ("Danau", danau),
    ("Laut", laut),
    ("Hutan", hutan),
    ("Mangrove", mangrove)
]:
    print(f"{nama}: {len(data)}")

print("\nTotal sampel gabungan:", len(samples))
print("Jumlah ID unik:", samples["sample_id"].nunique())


Jumlah fitur pada shapefile sumber:
Sawah: 50
Bangunan: 50
Danau: 31
Laut: 60
Hutan: 50
Mangrove: 50

Total sampel gabungan: 291
Jumlah ID unik: 291


In [79]:
import geopandas as gpd
import numpy as np
from sklearn.cluster import DBSCAN

# Pastikan data menggunakan GeoDataFrame
samples = samples.copy()

# Pastikan CRS tersedia dan ubah ke UTM untuk perhitungan jarak meter
if samples.crs is None:
    raise ValueError("CRS samples belum tersedia. Periksa CRS shapefile sumber.")

samples = samples.to_crs("EPSG:4326")
samples_utm = samples.to_crs("EPSG:32749")

# Ambil titik pusat setiap polygon
centroids = samples_utm.geometry.centroid

coords = np.column_stack([
    centroids.x.to_numpy(),
    centroids.y.to_numpy()
])

# Kelompokkan sampel berdasarkan jarak
dbscan = DBSCAN(eps=20000, min_samples=1)
samples["cluster"] = dbscan.fit_predict(coords) + 1

# Periksa hasil
print("Total sampel:", len(samples))
print("Jumlah cluster:", samples["cluster"].nunique())

print("\nJumlah sampel per cluster:")
print(samples["cluster"].value_counts().sort_index())

print("\nDistribusi kelas per cluster:")
print(samples.groupby(["cluster", "label"]).size().unstack(fill_value=0))

Total sampel: 291
Jumlah cluster: 3

Jumlah sampel per cluster:
cluster
1    235
2     50
3      6
Name: count, dtype: int64

Distribusi kelas per cluster:
label    bangunan  danau  hutan  laut  mangrove  sawah
cluster                                               
1              50     31      0    60        44     50
2               0      0     50     0         0      0
3               0      0      0     0         6      0


In [80]:
# Pastikan ID sampel tersedia dan unik
samples = samples.reset_index(drop=True).copy()

if "sample_id" not in samples.columns:
    samples["sample_id"] = range(len(samples))

# Pastikan kolom penting tersedia
print("Total sampel:", len(samples))
print("Jumlah ID unik:", samples["sample_id"].nunique())
print("Kolom tersedia:", samples.columns.tolist())

print("\nJumlah sampel per kelas:")
print(samples["label"].value_counts())

print("\nJumlah sampel per cluster:")
print(samples["cluster"].value_counts().sort_index())

Total sampel: 291
Jumlah ID unik: 291
Kolom tersedia: ['FID', 'geometry', 'label', 'sample_id', 'cluster']

Jumlah sampel per kelas:
label
laut        60
sawah       50
bangunan    50
hutan       50
mangrove    50
danau       31
Name: count, dtype: int64

Jumlah sampel per cluster:
cluster
1    235
2     50
3      6
Name: count, dtype: int64


In [81]:
import os
import geopandas as gpd

# Siapkan folder hasil
os.makedirs("hasil/ekstraksi", exist_ok=True)

# Pastikan CRS sesuai untuk OpenEO
samples = samples.to_crs("EPSG:4326").copy()

# Ambil maksimal 10 sampel pertama dari cluster pertama
cluster_id = sorted(samples["cluster"].unique())[0]

batch = (
    samples[samples["cluster"] == cluster_id]
    .head(10)
    .copy()
)

# Perbaiki geometri yang tidak valid
batch["geometry"] = batch.geometry.make_valid()

print("Cluster uji:", cluster_id)
print("Jumlah polygon:", len(batch))
print("Distribusi kelas:")
print(batch["label"].value_counts())

Cluster uji: 1
Jumlah polygon: 10
Distribusi kelas:
label
sawah    10
Name: count, dtype: int64


In [82]:
import openeo

# Band Sentinel-2 yang digunakan
bands = [
    "B02", "B03", "B04", "B05", "B06",
    "B07", "B08", "B8A", "B11", "B12"
]

tanggal = ["2025-08-01", "2025-09-01"]

# Batas wilayah berdasarkan 10 polygon uji
west, south, east, north = batch.total_bounds

spatial_extent = {
    "west": float(west),
    "south": float(south),
    "east": float(east),
    "north": float(north),
    "crs": "EPSG:4326"
}

# Muat data Sentinel-2 Level-2A
cube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=spatial_extent,
    temporal_extent=tanggal,
    bands=bands,
    max_cloud_cover=20
)

# Gabungkan pengamatan tanggal yang tersedia menggunakan median
median = cube.reduce_dimension(
    dimension="t",
    reducer="median"
)

# Samakan resolusi ke 20 meter
median = median.resample_spatial(
    resolution=20,
    projection="EPSG:32749"
)

print("Data Sentinel-2 berhasil disiapkan.")

Data Sentinel-2 berhasil disiapkan.


In [83]:
# Ambil band yang diperlukan
b03 = median.band("B03")
b04 = median.band("B04")
b08 = median.band("B08")
b11 = median.band("B11")

# Hitung indeks spektral
ndvi = ((b08 - b04) / (b08 + b04)).add_dimension(
    name="bands", label="NDVI", type="bands"
)

ndwi = ((b03 - b08) / (b03 + b08)).add_dimension(
    name="bands", label="NDWI", type="bands"
)

ndbi = ((b11 - b08) / (b11 + b08)).add_dimension(
    name="bands", label="NDBI", type="bands"
)

ndmi = ((b08 - b11) / (b08 + b11)).add_dimension(
    name="bands", label="NDMI", type="bands"
)

mndwi = ((b03 - b11) / (b03 + b11)).add_dimension(
    name="bands", label="MNDWI", type="bands"
)

# Gabungkan 10 band dan 5 indeks
features = median

for index_cube in [ndvi, ndwi, ndbi, ndmi, mndwi]:
    features = features.merge_cubes(index_cube)

print("10 band dan 5 indeks berhasil disiapkan.")

10 band dan 5 indeks berhasil disiapkan.


In [85]:
print(connection.describe_account())

{'info': {'oidc_userinfo': {'email': 'triswanti1395@gmail.com', 'email_verified': True, 'family_name': "Jannatul Ma'wa", 'given_name': 'Triswanti', 'name': "Triswanti Jannatul Ma'wa", 'preferred_username': 'triswanti1395@gmail.com', 'sub': '08868660-f1d0-4cae-bac7-ad9a53fa7209'}}, 'name': "Triswanti Jannatul Ma'wa", 'user_id': '08868660-f1d0-4cae-bac7-ad9a53fa7209'}


In [89]:
try:
    collections = connection.list_collections()

    print("Koneksi OpenEO berhasil!")
    print("Jumlah koleksi:", len(collections))

    for collection in collections[:10]:
        print(collection.get("id"), "-", collection.get("title"))

except Exception as e:
    print("Terjadi error:", e)

Koneksi OpenEO berhasil!
Jumlah koleksi: 137
SENTINEL1_GRD - Sentinel-1 SAR GRD: C-band Synthetic Aperture Radar Ground Range Detected.
COPERNICUS_30 - Copernicus Global 30 meter Digital Elevation Model dataset.
SENTINEL3_OLCI_L1B - Sentinel 3 OLCI
SENTINEL3_SLSTR - Sentinel 3 SLSTR
SENTINEL_5P_L2 - Sentinel 5 Precursor
LANDSAT_BIMONTHLY_MOSAIC - Landsat mosaic (bi-monthly, v1.0.1)
SENTINEL3_SYN_L2_SYN - Daily global Sentinel-3 Synergy products at 300m resolution combining OCLI and SLSTR bands.
SENTINEL3_SLSTR_L2_LST - Daily global Sentinel-3 Land surface temperature at 1km resolution
SENTINEL5P_L2_CO - Sentinel-5P TROPOMI Carbon Monoxide L2
SENTINEL5P_L2_NO2 - Sentinel-5P TROPOMI Nitrogendioxide L2


In [90]:
try:
    info = connection.describe_collection("SENTINEL2_L2A")

    print("Koleksi Sentinel-2 tersedia!")
    print("ID:", info.get("id"))
    print("Judul:", info.get("title"))

except Exception as e:
    print("Gagal memeriksa koleksi:", e)

Koleksi Sentinel-2 tersedia!
ID: SENTINEL2_L2A
Judul: Sentinel-2 L2A


In [91]:
# Ambil 10 sampel pertama
batch = samples.head(10).to_crs("EPSG:4326").copy()

west, south, east, north = batch.total_bounds

# Muat satu band Sentinel-2
cube_uji = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent={
        "west": float(west),
        "south": float(south),
        "east": float(east),
        "north": float(north),
        "crs": "EPSG:4326"
    },
    temporal_extent=["2025-08-01", "2025-09-01"],
    bands=["B04"],
    max_cloud_cover=20
)

# Ambil median waktu dan simpan sebagai GeoTIFF
uji = cube_uji.reduce_dimension(
    dimension="t",
    reducer="median"
)

job = uji.save_result(
    format="GTiff"
).create_job(
    title="Uji Ekspor Sentinel-2"
)

print("Job berhasil dibuat!")
print("Job ID:", job.job_id)
print("Status:", job.status())

Failed to parse API error response: [503] 'no available server\n' (headers: {'Content-Type': 'text/plain; charset=utf-8', 'X-Content-Type-Options': 'nosniff', 'Date': 'Sat, 10 Oct 2026 05:24:07 GMT', 'Content-Length': '20'})


OpenEoApiPlainError: [503] no available server
